# 01 · Data Audit

**Purpose.** Count and describe everything in `data/raw` before using it: files, metadata, labels, duplicates.

| | |
|---|---|
| **Reads** | `data/raw/*` |
| **Writes** | `reports/dataset_audit/inventory.csv`, `inventory_summary.csv`, `metadata_overview.csv` |
| **Run time (Colab T4)** | 5-10 min (lists ~43,000 files on Drive) |
| **Next notebook** | 02 · Label Taxonomy |

Run the cells from top to bottom. All shared logic lives in `src/dermalens/`; this
notebook only calls it and explains each decision.

## 0. Setup

The same cell starts every notebook. It (1) connects Google Drive on Colab,
(2) installs missing libraries on Colab, (3) makes `src/` importable,
(4) finds every project folder and (5) loads `configs/baseline.yaml`.

In [ ]:
import os
import sys
from pathlib import Path

# 1) On Colab, connect Google Drive so the project folder is visible.
try:
    from google.colab import drive                      # this import only works on Colab
    drive.mount("/content/drive")
    os.environ.setdefault("DERMALENS_PROJECT_DIR", "/content/drive/MyDrive/DermaLens_AI")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False                                    # local run: set DERMALENS_PROJECT_DIR yourself

# 2) Where the data lives (PROJECT_DIR) and where the code lives (CODE_DIR, normally the same folder).
PROJECT_DIR = Path(os.environ.get("DERMALENS_PROJECT_DIR", Path.cwd().parent))
CODE_DIR = Path(os.environ.get("DERMALENS_CODE_DIR", PROJECT_DIR))

# 3) Colab starts empty every session: install the few libraries it does not ship with.
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_DIR / "requirements.txt")], check=True)

# 4) Make `import dermalens` work.
sys.path.insert(0, str(CODE_DIR / "src"))

from dermalens.config import load_config
from dermalens.paths import ProjectPaths
from dermalens.utils import set_seed

# 5) Folders and settings.
paths = ProjectPaths(PROJECT_DIR).ensure()
CONFIG = load_config(os.environ.get("DERMALENS_CONFIG", CODE_DIR / "configs" / "baseline.yaml"))
set_seed(CONFIG["seed"])
print(paths.describe())

## 1. File inventory
Every file under `data/raw`, with its type and size. Nothing is opened yet.

In [ ]:
import pandas as pd
from dermalens.audit import inventory_files, summarise_inventory, skindisnet_mirror_check

DATASETS = [d.name for d in sorted(paths.raw.iterdir()) if d.is_dir()]
inventory = inventory_files(paths.raw, DATASETS)
inventory.to_csv(paths.audit / "inventory.csv", index=False)

summary = summarise_inventory(inventory)
summary.to_csv(paths.audit / "inventory_summary.csv", index=False)
display(summary)

Images per sub-folder. SkinDisNet's sub-folders matter: only one of them holds originals.

In [ ]:
images = inventory[inventory["kind"] == "image"]
display(images.groupby(["dataset", "subfolder"]).size().rename("images").reset_index())

## 2. SkinDisNet: how many photos are really unique?

`skindisnet1/Preprocessed` names files `AD_01.jpg`, `skindisnet2/Preprocessed` names the same photos
`AD (1).jpg`. The two `Augmented` folders are rotated/flipped copies. If both folder names match
one-to-one, the 26,943 files hold only **1,710** distinct photos.

In [ ]:
mirror = skindisnet_mirror_check(inventory)
print(mirror)
if mirror["matching_names"] == mirror["skindisnet2_originals"]:
    print("-> skindisnet1/Preprocessed mirrors skindisnet2/Preprocessed. Use skindisnet2 only;"
          " augmented folders are never used for evaluation.")

## 3. Metadata files: shape and columns
IDs are read as text so long numbers are not rounded.

In [ ]:
overview = []
for table_path in sorted(inventory.loc[inventory["suffix"] == ".csv", "rel_path"]):
    table = pd.read_csv(paths.raw / table_path, dtype=str, nrows=None)
    overview.append({"file": table_path, "rows": len(table), "columns": len(table.columns),
                     "column_names": ", ".join(table.columns[:12]) + (" ..." if len(table.columns) > 12 else "")})
overview = pd.DataFrame(overview)
overview.to_csv(paths.audit / "metadata_overview.csv", index=False)
display(overview)

## 4. DermaCon-IN: labels, subjects, quality

In [ ]:
derma = pd.read_csv(paths.raw / "dermaconin" / "METADATA" / "Skin_Metadata-1.csv", dtype={"Subject_ID": str})

label_counts = derma["Disease_label"].str.strip().value_counts()
print(f"{len(derma):,} images | {derma['Subject_ID'].nunique():,} subjects | {label_counts.size} diagnoses")
print(f"Diagnoses with fewer than 20 images: {(label_counts < 20).sum()} "
      f"(the long tail) | top 10 cover {label_counts.head(10).sum() / len(derma):.0%} of images")
print("Every Image_name unique:", derma["Image_name"].is_unique)
display(label_counts.head(15).rename("images").to_frame())
display(derma["Quality"].value_counts().rename("images").to_frame())

## 5. SkinDisNet: labels and patients

In [ ]:
skin = pd.read_csv(paths.raw / "skindisnet" / "skindisnet2" / "SkinDisNet_Metadata.csv", dtype=str)
print(f"{len(skin):,} images | {skin['Patient_id'].nunique()} patients")
print("Patients with more than one diagnosis:", int((skin.groupby("Patient_id")["Diagnosis"].nunique() > 1).sum()))
display(skin["Diagnosis"].value_counts().rename("images").to_frame())

## 6. MCSI: the healthy-skin source

In [ ]:
mcsi = pd.read_csv(paths.raw / "mcsi" / "extracted" / "metadata.csv", dtype=str)
display(mcsi["diagnostic"].value_counts().rename("images").to_frame())

## 7. SCIN: damaged case IDs (reference dataset only)

In version 1, SCIN reported 2,987 "duplicate" case IDs. The real cause: the CSV was opened and
re-saved in a spreadsheet tool, which turned 19-digit IDs into text like `-1.00E+18`. Many
different IDs collapsed into the same text. **Fix:** download `dataset_scin_cases.csv` and
`dataset_scin_labels.csv` again from the SCIN GitHub page and never open them in Excel/Sheets.

In [ ]:
from dermalens.manifest import count_damaged_ids

scin_cases_path = paths.raw / "scin" / "dataset_scin_cases.csv"
if scin_cases_path.exists():
    scin_cases = pd.read_csv(scin_cases_path, dtype={"case_id": str})
    damaged = count_damaged_ids(scin_cases["case_id"])
    print(f"SCIN rows: {len(scin_cases):,} | IDs in scientific notation (damaged): {damaged:,}")
    if damaged:
        print("-> Re-download the SCIN CSVs before using SCIN for anything.")
else:
    print("SCIN not present - fine, it is reference-only.")

## 8. What this audit tells us

| Finding | Consequence for the next notebooks |
|---|---|
| SkinDisNet's 26,943 files are 1,710 photos | Use `skindisnet2/Preprocessed` only (notebook 03) |
| DermaCon-IN has a long tail of rare diagnoses | Pick a few well-populated classes (notebook 02) |
| No source has a healthy class except MCSI | Healthy comes from one source: a shortcut risk (notebooks 04, 11) |
| DermaCon-IN groups photos by `Subject_ID`, SkinDisNet by `Patient_id` | Split by patient (notebook 05) |
| SCIN IDs are damaged | Re-download before any SCIN use |